# Combine the four final learning-rule runs

Upload the four ZIP files produced by the runner notebook. This notebook
verifies protocol compatibility, combines performance/SNR/cosine
tables, creates final figures, and downloads one final analysis ZIP.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

uploaded = files.upload()
INPUT_ROOT = Path("/content/final_comparison_inputs")
if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)
INPUT_ROOT.mkdir(parents=True)

for filename, content in uploaded.items():
    archive_path = INPUT_ROOT / filename
    archive_path.write_bytes(content)
    shutil.unpack_archive(str(archive_path), str(INPUT_ROOT / archive_path.stem))

print("Uploaded archives:", sorted(uploaded))


## 1. Load and validate every artifact

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

def load_csv_family(filename):
    paths = sorted(INPUT_ROOT.rglob(filename))
    if not paths:
        raise FileNotFoundError(f"No {filename} files were found.")
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["source_file"] = str(path)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

performance_df = load_csv_family("performance_summary.csv")
traces_df = load_csv_family("performance_traces.csv")
update_df = load_csv_family("update_metrics.csv")

required_rules = {
    "backprop",
    "feedback_alignment",
    "predictive_coding",
    "hebbian",
}
found_rules = set(performance_df["rule"])
missing_rules = required_rules - found_rules
if missing_rules:
    raise RuntimeError(f"Missing rule artifacts: {sorted(missing_rules)}")

for rule in required_rules:
    conditions = set(
        performance_df.loc[performance_df["rule"] == rule, "condition"]
    )
    if conditions != {"sequential", "interleaved"}:
        raise RuntimeError(
            f"{rule} has conditions {conditions}, expected both final conditions."
        )

if not (performance_df["architecture"] == "784-1000-10").all():
    raise RuntimeError("At least one run used the wrong architecture.")
if not np.allclose(performance_df["learning_rate"], 0.001):
    raise RuntimeError("At least one run used the wrong learning rate.")

duplicates = performance_df.duplicated(["rule", "condition"], keep=False)
if duplicates.any():
    display(performance_df.loc[duplicates])
    raise RuntimeError(
        "Duplicate rule/condition rows found. Upload exactly one final ZIP per rule."
    )

print("All four rules and both conditions passed protocol validation.")


## 2. Final performance table

In [ ]:
performance_columns = [
    "rule",
    "condition",
    "phase1_old_accuracy",
    "retained_old_accuracy",
    "forgetting",
    "normalized_forgetting",
    "new_class_accuracy",
    "balanced_final_accuracy",
    "optimizer",
]
final_performance = performance_df[performance_columns].sort_values(
    ["condition", "rule"]
)
display(
    final_performance.style.format(
        {
            "phase1_old_accuracy": "{:.2f}%",
            "retained_old_accuracy": "{:.2f}%",
            "forgetting": "{:.2f} pp",
            "normalized_forgetting": "{:.3f}",
            "new_class_accuracy": "{:.2f}%",
            "balanced_final_accuracy": "{:.2f}%",
        }
    )
)


## 3. Forgetting trajectories and endpoint comparison

In [ ]:
import matplotlib.pyplot as plt

traces_df["global_epoch"] = traces_df["recorded_epoch"]
traces_df.loc[traces_df["phase"] == 2, "global_epoch"] += 20

OUTPUT_DIR = Path("/content/final_comparative_analysis")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for axis, condition in zip(axes, ("sequential", "interleaved")):
    condition_df = traces_df[traces_df["condition"] == condition]
    for rule, rule_df in condition_df.groupby("rule"):
        axis.plot(
            rule_df["global_epoch"],
            rule_df["old_accuracy"],
            label=rule,
            linewidth=2,
        )
    axis.axvline(20.5, color="black", linestyle="--", alpha=0.6)
    axis.set_title(f"Old-task retention: {condition}")
    axis.set_xlabel("Recorded epoch")
    axis.set_ylim(-2, 102)
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Accuracy on digits 0-5 (%)")
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURE_DIR / "old_task_retention_curves.png", dpi=180)
plt.show()

endpoint = performance_df.sort_values(["condition", "rule"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for axis, condition in zip(axes, ("sequential", "interleaved")):
    subset = endpoint[endpoint["condition"] == condition]
    x = np.arange(len(subset))
    axis.bar(x - 0.2, subset["retained_old_accuracy"], width=0.4, label="Old")
    axis.bar(x + 0.2, subset["new_class_accuracy"], width=0.4, label="New")
    axis.set_xticks(x)
    axis.set_xticklabels(subset["rule"], rotation=30, ha="right")
    axis.set_title(f"Final performance: {condition}")
    axis.set_ylim(0, 105)
    axis.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Accuracy (%)")
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURE_DIR / "final_old_new_accuracy.png", dpi=180)
plt.show()


## 4. Layer-wise SNR and cosine

Statistical variation is summarized across fixed probe batches.
Hidden and output layers remain separate.


In [ ]:
checkpoint_order = [
    "initial",
    "phase1_end",
    "phase2_epoch_1",
    "phase2_epoch_5",
    "phase2_epoch_10",
    "phase2_epoch_20",
]
layer_styles = {
    "hidden_weight": ("-", "o"),
    "output_weight": ("--", "s"),
}

def plot_update_metric(metric, condition, probe_split, ylabel, filename):
    subset = update_df[
        (update_df["condition"] == condition)
        & (update_df["probe_split"] == probe_split)
    ].copy()
    subset["checkpoint_index"] = subset["checkpoint"].map(
        {name: i for i, name in enumerate(checkpoint_order)}
    )
    subset = subset.dropna(subset=["checkpoint_index"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    for axis, rule in zip(axes.ravel(), sorted(required_rules)):
        rule_df = subset[subset["rule"] == rule]
        for layer, (linestyle, marker) in layer_styles.items():
            layer_df = rule_df[rule_df["layer"] == layer].sort_values(
                "checkpoint_index"
            )
            axis.plot(
                layer_df["checkpoint_index"],
                layer_df[metric],
                linestyle=linestyle,
                marker=marker,
                label=layer,
            )
            if metric == "cosine_mean":
                sem = layer_df["cosine_sem"].fillna(0)
                axis.fill_between(
                    layer_df["checkpoint_index"],
                    layer_df[metric] - sem,
                    layer_df[metric] + sem,
                    alpha=0.15,
                )
        axis.set_title(rule)
        axis.grid(alpha=0.25)
        if metric == "cosine_mean":
            axis.axhline(0, color="black", linestyle=":", alpha=0.5)
            axis.set_ylim(-1.05, 1.05)
    for axis in axes[-1]:
        axis.set_xticks(range(len(checkpoint_order)))
        axis.set_xticklabels(checkpoint_order, rotation=50, ha="right")
    axes[0, 0].set_ylabel(ylabel)
    axes[1, 0].set_ylabel(ylabel)
    axes[0, 1].legend()
    fig.suptitle(f"{ylabel}: {condition}, {probe_split}")
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=180)
    plt.show()

for condition in ("sequential", "interleaved"):
    for probe_split in ("old_0_5", "new_6_9"):
        slug = f"{condition}_{probe_split}"
        plot_update_metric(
            "snr",
            condition,
            probe_split,
            "Update SNR",
            f"snr_{slug}.png",
        )
        plot_update_metric(
            "cosine_mean",
            condition,
            probe_split,
            "Cosine to BP descent",
            f"cosine_{slug}.png",
        )


## 5. Export the complete final analysis

In [ ]:
final_performance.to_csv(OUTPUT_DIR / "performance_summary.csv", index=False)
traces_df.to_csv(OUTPUT_DIR / "performance_traces.csv", index=False)
update_df.to_csv(OUTPUT_DIR / "update_metrics.csv", index=False)

sequential = final_performance[
    final_performance["condition"] == "sequential"
].sort_values("normalized_forgetting")
interleaved = final_performance[
    final_performance["condition"] == "interleaved"
].sort_values("normalized_forgetting")
report_lines = [
    "# Final comparative analysis",
    "",
    "Architecture: 784-1000-10; full MNIST; old 0-5, new 6-9; "
    "batch 32; learning rate 0.001; seed 0.",
    "",
    "## Sequential ranking by normalized forgetting",
]
for row in sequential.itertuples():
    report_lines.append(
        f"- {row.rule}: normalized forgetting "
        f"{row.normalized_forgetting:.3f}, retained old "
        f"{row.retained_old_accuracy:.2f}%, new "
        f"{row.new_class_accuracy:.2f}%."
    )
report_lines.extend(["", "## Interleaved control"])
for row in interleaved.itertuples():
    report_lines.append(
        f"- {row.rule}: normalized forgetting "
        f"{row.normalized_forgetting:.3f}, retained old "
        f"{row.retained_old_accuracy:.2f}%, new "
        f"{row.new_class_accuracy:.2f}%."
    )
report_lines.extend(
    [
        "",
        "## Limitations",
        "- Results use one seed and support descriptive conclusions only.",
        "- The first recorded epoch is a baseline for BP, FA, and Hebbian; "
        "PC updates internally during that pass.",
        "- PC update cosine uses the non-mutating JPC/Optax parameter delta; "
        "the other rules expose pre-optimizer learning directions.",
        "- SNR and cosine are signal-consistency and directional-alignment "
        "proxies, not literal statistical variance and bias.",
    ]
)
(OUTPUT_DIR / "ANALYSIS_REPORT.md").write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

final_zip = Path(
    shutil.make_archive(
        "/content/final_comparative_analysis_784_1000_10",
        "zip",
        root_dir=OUTPUT_DIR,
    )
)
print("Final analysis:", final_zip)
files.download(str(final_zip))
